# Evaluating An Agent with LangSmith

In this notebook it is demonstrated how to build a historical storytelling agent called **Snackable History** — an AI agent that explains complex events in short, engaging ways.

The agent is then tested using LangSmith

The workflow is as follows:

✅ Build the agent (web search + storytelling)  
✅ Create or reference a golden dataset of great responses  
✅ Define evaluation metrics (e.g. conciseness and engagement)  
✅ Run an experiment and analyse results  


Install Dependencies

In [2]:
!pip install -U \
    langchain==0.3.27 \
    langgraph==0.6.4 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-tavily==0.2.11

### Import Libraries

In [ ]:
from openai import AzureOpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

### Set up your API keys 


In [ ]:
import os
# LangSmith 
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "" # Change this to match the name of your LangSmith project
os.environ["LANGSMITH_API_KEY"] = "" # Add your LangSmith api key here

# Tavily
os.environ["TAVILY_API_KEY"] = "" # Add Tavily api key here

# OpenAI
os.environ['AZURE_OPENAI_API_KEY'] = '' #Add OpenAI api key here
os.environ['AZURE_OPENAI_ENDPOINT'] = ''
os.environ['OPENAI_API_VERSION'] = '2024-08-01-preview'

### Create a tool instance using your Tavily API key.

In [16]:
from langchain_tavily import TavilySearch
web_search_tool = TavilySearch(max_results=1)

### Create an prompt for the agent. The agent will transform dry history facts into vivid, short, and memorable answers.




In [9]:
prompt = """ You are a storyteling agent turning dry history facts into vivid, short and memorable answers.

Your job is to answer the question using the context provided. 

Focus on accuracy and engaging storytelling.

Question: {question}

Context: {context}

Answer: """


print("Prompt Template:\n", prompt)

Prompt Template:
  You are a storyteling agent turning dry history facts into vivid, short and memorable answers.

Your job is to answer the question using the context provided. 

Focus on accuracy and engaging storytelling.

Question: {question}

Context: {context}

Answer: 


### Wrap the AzureOpenAI client so LangSmith can trace its use

In [11]:
openai_client = wrap_openai(AzureOpenAI())

Create a function that uses the Tavily search tool to search for the user’s topic and use the traceable decorator. This function:

* Sends the question to Tavily
* Collect the top search results
* Join them into a single text block to use as context

In [22]:
@traceable
def search(question):
    web_docs = web_search_tool.invoke({"query": question}).get("results", [])
    web_results = "\n".join([d["content"] for d in web_docs]) 
    return web_results

Define a function that asks OpenAI to summarize the context into a user-friendly answer.

In [19]:
@traceable
def explain(question, context):
    formatted = prompt.format(question=question, context=context)

    completion = openai_client.chat.completions.create(
        messages=[
            {"role": "system", "content": formatted},
            {"role": "user", "content": question},
        ],
        model="gpt-4.1-mini",
    )
    return completion.choices[0].message.content

Write a function called that combines both search and explain steps to deliver the full experience. 

In [20]:
@traceable
def snackable_history(question):
    context = search(question)
    answer = explain(question, context)
    return answer

**Test the agent**

In [23]:
question = "Explain how the blue LED was created"
print(snackable_history(question))

In the vibrant world of light, one inventor’s determination shined through like a beacon. In 1988, Shuji Nakamura embarked on a quest to conquer the elusive blue LED, a challenge that had stumped many before him. By 1990, he struck gold with high-quality gallium nitride. 

Nakamura’s innovative approach was like a symphony, where the crystal's creation was the crescendo. He began by growing a thin layer of gallium nitride at a cool temperature, setting the stage for what was to come. Then, with a fiery passion, he raised the temperature for subsequent layers, coaxing them into perfect harmony. 

Within this layered masterpiece, positive holes and negative electrons danced energetically. When electricity flowed, they met in the active layer, melding together in a brilliant flash—light was born! The magic was in the materials; those special semiconductors played the crucial role in creating the vibrant blue light that would illuminate the world. This groundbreaking invention not only cha